# CSMorgan — Fine-tune SimulaMet adapter & submit

**One-shot pipeline.** Switch runtime to **A100** (Runtime → Change runtime type → A100), then Runtime → Run all.

Steps: install → login → mount Drive → train ~45 min → push adapter → push inference files → validate → submit.

Total wallclock: ~70 min on A100.

Repo created: `sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2`

### 1. Install dependencies (~3 min)

In [ ]:
%%capture
!pip install -q -U \
    'transformers>=4.45.0' 'accelerate>=0.34.0' 'peft>=0.13.0' \
    'bitsandbytes>=0.43.0' ms-swift==3.8.0 qwen_vl_utils==0.0.11 \
    'datasets>=2.20.0' 'evaluate>=0.4.3' sacrebleu rouge_score nltk medvqa huggingface_hub
import nltk
for p in ['wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(p, quiet=True)
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → GPU.'
print('GPU:', torch.cuda.get_device_name(0))

### 2. Hugging Face login (write-scope token required)

In [ ]:
from huggingface_hub import notebook_login, whoami
notebook_login()

In [ ]:
from huggingface_hub import whoami
info = whoami()
assert info['name'] == 'sageofai', f"Expected sageofai, got {info['name']}"
print('Logged in as:', info['name'])

Logged in as: sageofai


### 3. Mount Google Drive (so checkpoints survive Colab disconnects)
Training writes to `MyDrive/csmorgan_v2_ft/`. If Colab disconnects mid-training, re-run the training cell and it will auto-resume from the last checkpoint.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

OUT_DIR = '/content/drive/MyDrive/csmorgan_v2_ft'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output dir: {OUT_DIR}')

existing = [d for d in os.listdir(OUT_DIR) if d.startswith('checkpoint-')] \
           if os.path.isdir(OUT_DIR) else []
if existing:
    print(f'Found {len(existing)} existing checkpoint(s) - training will RESUME.')
else:
    print('No checkpoints. Fresh training run.')

Mounted at /content/drive
Output dir: /content/drive/MyDrive/csmorgan_v2_ft
Found 2 existing checkpoint(s) - training will RESUME.


### 4. Fine-tune SimulaMet's adapter (~45 min on A100)
Loads `SimulaMet/Qwen2.5-VL-KvasirVQA-x1-ft` as the initial LoRA weights, continues training for 1 epoch on 3000 randomly-sampled Kvasir-VQA-x1 examples, saves new weights to Drive.

Tweak `NUM_SAMPLES` if you want a different time budget. 500 ≈ 8 min smoke test, 3000 ≈ 45 min, 5000 ≈ 75 min.

In [ ]:
NUM_SAMPLES = 3000
NUM_EPOCHS = 1
LEARNING_RATE = 5e-5
MAX_LENGTH = 1024
GRAD_ACCUM = 8

import torch
from transformers import (
    AutoProcessor, Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig, TrainingArguments, Trainer,
)
from peft import PeftModel, prepare_model_for_kbit_training
from datasets import load_dataset

print('[1/5] Loading Qwen2.5-VL-7B in 4-bit nf4...')
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen2.5-VL-7B-Instruct',
    quantization_config=bnb,
    dtype=torch.bfloat16, # Changed from torch_dtype to dtype
    device_map='auto',
    attn_implementation='sdpa',
)
base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)

print('[2/5] Loading SimulaMet adapter as trainable initial weights...')
model = PeftModel.from_pretrained(
    base, 'SimulaMet/Qwen2.5-VL-KvasirVQA-x1-ft', is_trainable=True,
)
model.print_trainable_parameters()

print('[3/5] Loading processor and dataset...')
processor = AutoProcessor.from_pretrained('Qwen/Qwen2.5-VL-7B-Instruct')
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

ds = load_dataset('SimulaMet/Kvasir-VQA-x1', split='train') \
        .shuffle(seed=42).select(range(NUM_SAMPLES))
print(f'   training on {len(ds)} samples')

def _txt(v):
    if v is None: return ''
    if isinstance(v, str): return v.strip()
    if isinstance(v, (list, tuple)): return ' '.join(_txt(x) for x in v).strip()
    return str(v).strip()

def collate(batch):
    msgs, imgs = [], []
    for ex in batch:
        img = ex['image']
        if hasattr(img, 'convert') and img.mode != 'RGB':
            img = img.convert('RGB')
        imgs.append([img])
        msgs.append([
            {'role': 'user', 'content': [
                {'type': 'image'},
                {'type': 'text', 'text': _txt(ex['question'])},
            ]},
            {'role': 'assistant', 'content': [
                {'type': 'text', 'text': _txt(ex['answer'])},
            ]},
        ])
    texts = [processor.apply_chat_template(m, tokenize=False) for m in msgs]
    inp = processor(
        text=texts, images=imgs, return_tensors='pt',
        padding=True, truncation=False, max_length=MAX_LENGTH, # Changed truncation to False
    )
    labels = inp['input_ids'].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    img_pad = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    if isinstance(img_pad, int) and img_pad >= 0:
        labels[labels == img_pad] = -100
    inp['labels'] = labels
    return inp

print('[4/5] Setting up Trainer...')
args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to='none',
    remove_unused_columns=False,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    seed=42,
    dataloader_pin_memory=False,
)

# Auto-resume if checkpoint exists
resume = None
if os.path.isdir(OUT_DIR):
    cks = sorted([d for d in os.listdir(OUT_DIR) if d.startswith('checkpoint-')],
                 key=lambda d: int(d.split('-')[-1]))
    if cks:
        resume = os.path.join(OUT_DIR, cks[-1])
        print(f'   resuming from {resume}')

trainer = Trainer(
    model=model, args=args, train_dataset=ds,
    data_collator=collate, tokenizer=processor.tokenizer,
)

print('[5/5] Training...')
trainer.train(resume_from_checkpoint=resume)

final_dir = f'{OUT_DIR}/final'
os.makedirs(final_dir, exist_ok=True)
model.save_pretrained(final_dir)
processor.save_pretrained(final_dir)
print(f'✅ Done. Adapter saved to {final_dir}')
print('   Files:', os.listdir(final_dir))

[1/5] Loading Qwen2.5-VL-7B in 4-bit nf4...


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

[2/5] Loading SimulaMet adapter as trainable initial weights...


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.visual.blocks.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.0.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.1.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.1.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.down_proj.lora_A.default.weight', 'base_model.mod

trainable params: 47,589,376 || all params: 8,339,756,032 || trainable%: 0.5706
[3/5] Loading processor and dataset...


...

   training on 3000 samples
[4/5] Setting up Trainer...
[5/5] Training...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2752: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
10,5.138700
20,1.899900
30,0.966200
40,0.859400
50,0.840500
60,0.782000
70,0.726800
80,0.721300
90,0.701700
100,0.669300


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2752: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


✅ Done. Adapter saved to /content/drive/MyDrive/csmorgan_v2_ft/final
   Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'preprocessor_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json', 'vocab.json', 'merges.txt', 'tokenizer.json', 'video_preprocessor_config.json']


### 5. Push the trained adapter to HF (~1 min)

In [ ]:
from huggingface_hub import HfApi, create_repo

DST = 'sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2'
api = HfApi()
create_repo(DST, repo_type='model', exist_ok=True, private=False)

final_dir = f'{OUT_DIR}/final'
assert os.path.isdir(final_dir), f'No trained adapter at {final_dir}'

print(f'Uploading {final_dir}/ → {DST} ...')
api.upload_folder(
    folder_path=final_dir,
    repo_id=DST, repo_type='model',
    commit_message='v2: continued QLoRA on SimulaMet adapter',
)
print(f'✅ Adapter pushed → https://huggingface.co/{DST}')

Uploading /content/drive/MyDrive/csmorgan_v2_ft/final/ → sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2 ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 54.9kB /  190MB            

  ...2_ft/final/tokenizer.json:  30%|###       | 3.46MB / 11.4MB            

✅ Adapter pushed → https://huggingface.co/sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2


### 6. Copy `normalization.py`, `answer_bank.json`, `requirements.txt` from your v1 repo to v2
Reuses the working files you already have on HF — no rebuild needed.

In [ ]:
from huggingface_hub import hf_hub_download

SRC = 'sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1'

for fn in ['normalization.py', 'answer_bank.json', 'requirements.txt']:
    print(f'  fetching {fn} from {SRC} ...')
    path = hf_hub_download(repo_id=SRC, filename=fn, repo_type='model')
    api.upload_file(
        path_or_fileobj=path, path_in_repo=fn,
        repo_id=DST, repo_type='model',
        commit_message=f'v2: copy {fn} from v1',
    )
    print(f'  ✓ pushed {fn}')
print(f'\n✅ Companion files in place at https://huggingface.co/{DST}')

  fetching normalization.py from sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1 ...


normalization.py: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


  ✓ pushed normalization.py
  fetching answer_bank.json from sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1 ...


answer_bank.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

...

  ...a84de02c/answer_bank.json:  18%|#8        | 10.6MB / 58.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


  ✓ pushed answer_bank.json
  fetching requirements.txt from sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1 ...


requirements.txt:   0%|          | 0.00/672 [00:00<?, ?B/s]

  ✓ pushed requirements.txt

✅ Companion files in place at https://huggingface.co/sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2


### 7. Push `submission_task1.py` to v2
Downloads the working v1 script, swaps the repo-ID references from v1 → v2 (so it loads YOUR new adapter), pushes.

In [ ]:
v1_script = hf_hub_download(repo_id=SRC, filename='submission_task1.py', repo_type='model')
with open(v1_script) as f:
    code = f.read()
code = code.replace(
    '"sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1"',
    f'"{DST}"',
)
with open('/content/submission_task1.py', 'w') as f:
    f.write(code)

api.upload_file(
    path_or_fileobj='/content/submission_task1.py',
    path_in_repo='submission_task1.py',
    repo_id=DST, repo_type='model',
    commit_message='v2: submission_task1.py loading v2 adapter',
)
print(f'✅ submission_task1.py pushed (now loads {DST} adapter)')

submission_task1.py: 0.00B [00:00, ?B/s]

✅ submission_task1.py pushed (now loads sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2 adapter)


### 8. `medvqa validate` (~10 min on the org's A100)
Dry run, free, doesn't consume a submission slot. Inspect the metrics + FLAGS section, compare to v1's `rouge1=0.5423`.

In [ ]:
!medvqa validate --competition=gi-2026 --task=1 --repo_id=sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2

🌟 ImageCLEFmed-MEDVQA-GI-2026 🌟 https://github.com/simula/ImageCLEFmed-MEDVQA-GI-2026
🔍 Subtask 1: Clinically Relevant Visual Question Answering
👀 Analyzing submission repository: sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2 👀
⚠️⚠️ Not logged in to HuggingFace! Please get your login token from https://huggingface.co/settings/tokens 🌐

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Loaded as

### 9. Real submission
**Only run after Phase 8 metrics beat v1's 0.5423.** If not, leave v1's submission as your best.
Flip `CONFIRM = True`, then run.

In [ ]:
CONFIRM = True

if CONFIRM:
    !medvqa validate_and_submit --competition=gi-2026 --task=1 --repo_id=sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2
else:
    print('CONFIRM is False. Inspect Phase 8 output, then set CONFIRM=True and re-run.')

🌟 ImageCLEFmed-MEDVQA-GI-2026 🌟 https://github.com/simula/ImageCLEFmed-MEDVQA-GI-2026
🔍 Subtask 1: Clinically Relevant Visual Question Answering
👀 Analyzing submission repository: sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2 👀
Logged in to HuggingFace as: sageofai
Loaded as API: https://simulamet-medvqa-gi-2026.hf.space ✔
💓 Communicating with the Submission Server: Ping!
Pong! Submission server is alive! 😊
Fetching 2 files: 100% 2/2 [00:00<00:00, 30283.78it/s]
📦 Making sure of the minimum requirements to run the script 📦
📦 Installing requirements from the submission repo: sageofai/Qwen25VL-MEDVQA-GI-S1-subtask1-v2/requirements.txt
2026-05-18 14:52:25.312103: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-18 14:52:25.384356: I tensorflow/core/platform/cpu_feature_gu